# Hybrid AI Recommender — Grupa 3: Retrieval (demo v1)

Za kandidata pronalazimo top k semantički najsličnijih oglasa. Ovaj notebook pokazuje:
1. kako Sentence Transformer mjeri sličnost značenja,
2. kako iz `NormalizedProfile` objekata slažemo tekst za retrieval (s ESCO labelama),
3. FAISS pretragu i izlaz u formatu `RetrievalResult` za Grupu 4,
4. koliko pomaže ESCO normalizacija i kako se ponašaju različiti encoder modeli.

In [4]:
!pip install -q faiss-cpu sentence-transformers

import faiss
import sentence_transformers
print("faiss:", faiss.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

faiss: 1.15.1
sentence-transformers: 6.2.0



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Za kandidata pronalazimo top k semantički najsličnijih oglasa: profil → retrieval text → Sentence Transformer (embedding) → FAISS baza (vektori) → sličnost.

Sentence Transformer pretvara tekst u vektor od 384 broja. Tekstovi sličnog značenja dobiju bliske vektore, a blizinu mjerimo kosinusnom sličnošću (koliko 2 vektora gledaju u istom smjeru).

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

recenice = [
    "developed REST APIs using Python",
    "experience with backend web services",
    "FastAPI experience preferred",
    "designing mobile screens in Figma",
]

embeddings = model.encode(recenice)
similarity = util.cos_sim(embeddings, embeddings)
for i in range(1, len(recenice)):
    print(f"{similarity[0][i]:.2f}  '{recenice[0]}'  vs  '{recenice[i]}'")

C:\Users\PC\Documents\faks\diplomski\diplproj\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\PC\Documents\faks\diplomski\diplproj\.venv\Lib\site-packages\huggingface

0.37  'developed REST APIs using Python'  vs  'experience with backend web services'
0.25  'developed REST APIs using Python'  vs  'FastAPI experience preferred'
0.02  'developed REST APIs using Python'  vs  'designing mobile screens in Figma'


„REST APIs“ i „backend web services“ nemaju zajedničkih riječi, a sličnost im je 0.37. Figma rečenica ima 0.02. Model prepoznaje značenje, ne samo riječi.
FastAPI dobiva manje (0.25) jer je za model rijedak pojam. Zato je ESCO normalizacija korisna prije retrievala.

## Podaci u formatu NormalizedProfile

Originalni profil je pod `profile`, ESCO normalizacija pod `normalized` (pravilo „add, never overwrite”). Koristi se skraćena verzija iz `project_schemas.json`, samo polja potrebna za retrieval.

`normalized_label` je ESCO `preferred_label`. Vještina koju Grupa 2 ne uspije povezati (`UNMAPPED`) nema ESCO koncept, pa za nju zadržavamo izvorni pojam.

In [6]:
def skill(term, esco_label=None, status="UNMAPPED"):
    if esco_label is None:
        status = "UNMAPPED"
    return {
        "source_term": term,
        "normalized_label": esco_label if esco_label else term,
        "esco_concept": {"preferred_label": esco_label} if esco_label else None,
        "mapping": {"status": status},
    }

candidate = {
    "entity_type": "candidate",
    "entity_id": "cand_017",
    "profile": {
        "experience": [{"description": "Developed machine-learning models for biomedical time series"}],
        "preferences": {
            "target_roles": ["machine learning engineer", "data scientist"],
            "interests": ["machine learning", "data science"],
        },
    },
    "normalized": {"skills": [
        skill("Python", "Python (computer programming)", "EXACT"),
        skill("PyTorch", "machine learning", "RELATED"),
        skill("SQL", "SQL", "EXACT"),
        skill("FastAPI", "web programming", "RELATED"),
        skill("Hugging Face"),  # UNMAPPED: nema ESCO koncepta, ostaje izvorni pojam
    ]},
}

jobs = [
    {"entity_type": "job", "entity_id": "job_284",
     "profile": {"title": "Machine Learning Intern",
                 "education": {"level": "msc", "field": "Computer Science"},
                 "languages": [],
                 "location": {"city": "Zagreb"}, "remote_mode": {"value": "hybrid"}},
     "normalized": {"skills": [skill("Python", "Python (computer programming)", "EXACT"),
                               skill("SQL", "SQL", "EXACT"),
                               skill("Docker", "Docker", "EXACT")]}},

    {"entity_type": "job", "entity_id": "job_102",
     "profile": {"title": "Junior Data Scientist", "education": None,
                 "languages": [{"language": "German", "level": "B2", "modality": "required"}],
                 "location": {"city": "Munich"}, "remote_mode": {"value": "onsite"}},
     "normalized": {"skills": [skill("Python", "Python (computer programming)", "EXACT"),
                               skill("machine learning", "machine learning", "EXACT"),
                               skill("statistics", "statistics", "EXACT")]}},

    {"entity_type": "job", "entity_id": "job_115",
     "profile": {"title": "Python Backend Intern", "education": None,
                 "languages": [],
                 "location": {"city": "Split"}, "remote_mode": {"value": "remote"}},
     "normalized": {"skills": [skill("Python", "Python (computer programming)", "EXACT"),
                               skill("FastAPI", "web programming", "RELATED"),
                               skill("PostgreSQL", "PostgreSQL", "EXACT")]}},

    {"entity_type": "job", "entity_id": "job_163",
     "profile": {"title": "Data Analyst Intern", "education": {"level": "bsc", "field": "Economics"},
                 "languages": [],
                 "location": {"city": "Zagreb"}, "remote_mode": {"value": "onsite"}},
     "normalized": {"skills": [skill("SQL", "SQL", "EXACT"),
                               skill("Excel", "use spreadsheets software", "RELATED"),
                               skill("Power BI", "data visualisation software", "RELATED")]}},

    {"entity_type": "job", "entity_id": "job_226",
     "profile": {"title": "UX/UI Design Intern", "education": None,
                 "languages": [],
                 "location": {"city": "Zagreb"}, "remote_mode": {"value": "hybrid"}},
     "normalized": {"skills": [skill("Figma", "use interface design software", "RELATED"),
                               skill("user research", "conduct user research", "RELATED")]}},
]

print("kandidat:", candidate["entity_id"])
for job in jobs:
    print(job["entity_id"], "-", job["profile"]["title"])

kandidat: cand_017
job_284 - Machine Learning Intern
job_102 - Junior Data Scientist
job_115 - Python Backend Intern
job_163 - Data Analyst Intern
job_226 - UX/UI Design Intern


Imamo 1 kandidata i 5 oglasa različite sličnosti: dva ML/data science oglasa, dva djelomično povezana (backend, analitika) i jedan nepovezan (dizajn). Tako unaprijed znamo kakav poredak očekujemo i možemo provjeriti radi li retrieval.

## Tekst za retrieval

Iz svakog profila slažemo kratak tekst koji ide u encoder. U tekst ulazi samo ono što opisuje **sadržaj posla**: uloga, vještine, iskustvo, obrazovanje. Lokacija, remote mode i jezik namjerno su izostavljeni: kandidat ih nema u istom obliku, pa bi samo dodavali šum u sličnost, a ionako su to tvrdi uvjeti koje provjerava Grupa 4.

Parametar `use_esco` bira između ESCO labela i sirovih pojmova, kako bismo mogli usporediti koliko normalizacija pomaže.

In [7]:
def skills_text(entity, use_esco=True):
    key = "normalized_label" if use_esco else "source_term"
    return ", ".join(s[key] for s in entity["normalized"]["skills"])

def candidate_text(c, use_esco=True):
    p = c["profile"]
    prefs = p["preferences"]
    return "\n".join([
        f"Target roles: {', '.join(prefs['target_roles'])}.",
        f"Skills: {skills_text(c, use_esco)}.",
        f"Experience: {'; '.join(e['description'] for e in p['experience'])}.",
        f"Interests: {', '.join(prefs['interests'])}.",
    ])

def job_text(j, use_esco=True):
    p = j["profile"]
    lines = [f"Role: {p['title']}.", f"Skills: {skills_text(j, use_esco)}."]
    if p.get("education"):
        edu = p["education"]
        lines.append(f"Education: {edu['level']} {edu['field']}.")
    return "\n".join(lines)

print(candidate_text(candidate))
for job in jobs:
    print("\n---", job["entity_id"])
    print(job_text(job))

Target roles: machine learning engineer, data scientist.
Skills: Python (computer programming), machine learning, SQL, web programming, Hugging Face.
Experience: Developed machine-learning models for biomedical time series.
Interests: machine learning, data science.

--- job_284
Role: Machine Learning Intern.
Skills: Python (computer programming), SQL, Docker.
Education: msc Computer Science.

--- job_102
Role: Junior Data Scientist.
Skills: Python (computer programming), machine learning, statistics.

--- job_115
Role: Python Backend Intern.
Skills: Python (computer programming), web programming, PostgreSQL.

--- job_163
Role: Data Analyst Intern.
Skills: SQL, use spreadsheets software, data visualisation software.
Education: bsc Economics.

--- job_226
Role: UX/UI Design Intern.
Skills: use interface design software, conduct user research.


## FAISS indeks i pretraga

Embeddinge oglasa spremamo u FAISS indeks. Vektore normaliziramo, pa je skalarni produkt (`IndexFlatIP`) jednak kosinusnoj sličnosti. Za kandidata napravimo embedding i tražimo oglase s najvećom sličnošću.

Sve je složeno u funkcije koje primaju model, oglase i indeks kao argumente, tako da isti kod možemo vrtjeti za različite modele i kasnije ga spojiti s Grupom 4.

In [8]:
import faiss

def build_index(model, jobs, use_esco=True):
    emb = model.encode([job_text(j, use_esco) for j in jobs], normalize_embeddings=True)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index

def search(model, index, jobs, candidate, k=50, use_esco=True):
    k = min(k, len(jobs))
    query = model.encode([candidate_text(candidate, use_esco)], normalize_embeddings=True)
    scores, positions = index.search(query, k)
    return [(jobs[pos], float(score)) for score, pos in zip(scores[0], positions[0])]

def print_ranking(hits):
    for rank, (job, score) in enumerate(hits, start=1):
        print(f"{rank}. {score:.2f}  {job['entity_id']}  {job['profile']['title']}")

index = build_index(model, jobs)
print_ranking(search(model, index, jobs, candidate, k=5))

1. 0.71  job_102  Junior Data Scientist
2. 0.71  job_284  Machine Learning Intern
3. 0.55  job_115  Python Backend Intern
4. 0.54  job_163  Data Analyst Intern
5. 0.33  job_226  UX/UI Design Intern


Očekujemo ML / data science oglase na vrhu, djelomično povezane u sredini, a dizajn na dnu.

Retrieval gleda samo sličnost: job_102 traži njemački B2 (`modality: required`) i to retrieval namjerno ignorira. Takve uvjete provjerava Grupa 4 u constraint validatoru.

## Koliko pomaže ESCO normalizacija?

Isti retrieval pokrećemo dva puta: jednom sa sirovim pojmovima iz CV-a i oglasa (`source_term`), jednom s ESCO labelama (`normalized_label`).

In [9]:
def compare_esco(model, jobs, candidate, k=5):
    rows = {}
    for use_esco, name in [(False, "sirovi pojmovi"), (True, "ESCO labele")]:
        idx = build_index(model, jobs, use_esco)
        for rank, (job, score) in enumerate(search(model, idx, jobs, candidate, k, use_esco), start=1):
            rows.setdefault(job["entity_id"], {"title": job["profile"]["title"]})[name] = f"{rank}. ({score:.2f})"
    print(f"{'oglas':<9} {'naslov':<26} {'sirovi pojmovi':<16} {'ESCO labele':<16}")
    for job_id, r in sorted(rows.items(), key=lambda x: x[1]["ESCO labele"]):
        print(f"{job_id:<9} {r['title']:<26} {r['sirovi pojmovi']:<16} {r['ESCO labele']:<16}")

compare_esco(model, jobs, candidate)

oglas     naslov                     sirovi pojmovi   ESCO labele     
job_102   Junior Data Scientist      2. (0.68)        1. (0.71)       
job_284   Machine Learning Intern    1. (0.69)        2. (0.71)       
job_115   Python Backend Intern      4. (0.51)        3. (0.55)       
job_163   Data Analyst Intern        3. (0.51)        4. (0.54)       
job_226   UX/UI Design Intern        5. (0.31)        5. (0.33)       


Najzanimljiviji su RELATED parovi: npr. „PyTorch” postaje „machine learning”, a „FastAPI” postaje „web programming”. Rijetke alate model slabo razumije, a ESCO ih prevodi u pojmove koje poznaje, pa bi se razlika trebala vidjeti upravo kod job_102 i job_115.

## Usporedba encoder modela

Po planu Grupa 3 uspoređuje encodere. Na istom primjeru pokrećemo tri modela i gledamo dvije stvari:
- **top-2 točno**: jesu li oba ML / data science oglasa (job_102, job_284) na prva dva mjesta,
- **razmak**: prosječna sličnost relevantnih oglasa minus sličnost nepovezanog (job_226). Veći razmak znači da model jasnije razdvaja dobre od loših oglasa.

Na 5 oglasa ovo je samo ilustracija; prava usporedba radi se na stvarnom skupu kad Grupa 1 i 2 daju podatke.

In [10]:
ENCODERS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    "sentence-transformers/all-mpnet-base-v2",
]
RELEVANT = {"job_102", "job_284"}
IRRELEVANT = "job_226"

models = {}
print(f"{'model':<48} {'dim':>4}  {'top-2':<6} {'razmak':>6}  poredak")
for name in ENCODERS:
    m = models[name] = SentenceTransformer(name)
    idx = build_index(m, jobs)
    hits = search(m, idx, jobs, candidate, k=5)
    scores = {job["entity_id"]: s for job, s in hits}
    top2_ok = {job["entity_id"] for job, _ in hits[:2]} == RELEVANT
    gap = sum(scores[j] for j in RELEVANT) / len(RELEVANT) - scores[IRRELEVANT]
    order = " > ".join(job["entity_id"].replace("job_", "") for job, _ in hits)
    print(f"{name:<48} {idx.d:>4}  {'da' if top2_ok else 'ne':<6} {gap:>6.2f}  {order}")

model                                             dim  top-2  razmak  poredak


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2496.02it/s]


sentence-transformers/all-MiniLM-L6-v2            384  da       0.38  102 > 284 > 115 > 163 > 226


C:\Users\PC\Documents\faks\diplomski\diplproj\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--multi-qa-MiniLM-L6-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3972.61it/s]


sentence-transformers/multi-qa-MiniLM-L6-cos-v1   384  da       0.31  102 > 284 > 115 > 163 > 226


C:\Users\PC\Documents\faks\diplomski\diplproj\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1290.17it/s]


sentence-transformers/all-mpnet-base-v2           768  da       0.31  102 > 284 > 163 > 115 > 226


## Izlaz za Grupu 4: RetrievalResult

Funkcija `retrieve` vraća objekt u dogovorenom formatu. Grupa 4 za svaki oglas dobiva `job_id`, `semantic_similarity` (to im je jedan od featurea) i `retrieval_rank`.

In [11]:
import json

ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

def retrieve(candidate, jobs, model, index, encoder_name, k=50):
    hits = search(model, index, jobs, candidate, k)
    return {
        "schema_version": "1.0",
        "candidate_id": candidate["entity_id"],
        "encoder_model": encoder_name,
        "k": len(hits),
        "results": [
            {
                "job_id": job["entity_id"],
                "semantic_similarity": round(score, 4),
                "retrieval_rank": rank,
            }
            for rank, (job, score) in enumerate(hits, start=1)
        ],
    }

result = retrieve(candidate, jobs, model, index, ENCODER_NAME, k=5)
print(json.dumps(result, indent=2))

{
  "schema_version": "1.0",
  "candidate_id": "cand_017",
  "encoder_model": "sentence-transformers/all-MiniLM-L6-v2",
  "k": 5,
  "results": [
    {
      "job_id": "job_102",
      "semantic_similarity": 0.7117,
      "retrieval_rank": 1
    },
    {
      "job_id": "job_284",
      "semantic_similarity": 0.7052,
      "retrieval_rank": 2
    },
    {
      "job_id": "job_115",
      "semantic_similarity": 0.5531,
      "retrieval_rank": 3
    },
    {
      "job_id": "job_163",
      "semantic_similarity": 0.5379,
      "retrieval_rank": 4
    },
    {
      "job_id": "job_226",
      "semantic_similarity": 0.3258,
      "retrieval_rank": 5
    }
  ]
}


## Sljedeći koraci

- Zamijeniti ručne primjere pravim `NormalizedProfile` izlazom Grupe 2 (do tada primjerima iz `project_schemas.json`).
- Indeksirati svih ~500 oglasa iz Hugging Face skupa i vraćati top 50.
- Odlučiti FAISS ili Qdrant: FAISS je dovoljan za ovu veličinu, Qdrant donosi perzistenciju i filtriranje kroz Docker.
- Uspoređivati encodere na većem skupu s ručno označenim relevantnim oglasima (npr. precision@k, recall@k).